# 🗺️ Txoko — Descarga Google Maps Places API (2 fases)
### Reto Inetum · Bootcamp BBK The Bridge · Equipo 4

**Estrategia de coste documentada para el jurado:**

| Fase | Endpoint | Peticiones | Coste est. |
|---|---|---|---|
| 1 — Text Search | todos los lugares | 4.624 | ~$79 |
| 2 — Place Details | solo HIGH + MEDIUM | ~2.800 | ~$47 |
| **TOTAL** | | | **~$126 ✅** |

Crédito gratuito Google: **$200/mes** → margen de seguridad: ~$74

> La clave de API se carga desde `.env` — nunca hardcodeada.  
> El `.env` debe estar en el `.gitignore` del proyecto.


## 0. Dependencias

In [ ]:
# Descomentar si es necesario:
# !pip install python-dotenv requests pandas numpy tqdm
print("✓ Librerías listas")

## 1. Imports y carga de API Key

In [11]:
import os, json, time
import pandas as pd
import numpy as np
import requests
from math import radians, sin, cos, sqrt, atan2
from tqdm.auto import tqdm
from dotenv import load_dotenv

# Carga la API Key desde el archivo .env
# Contenido del .env:  GOOGLE_MAPS_API_KEY=AIzaSy_tu_clave
load_dotenv()
API_KEY = os.getenv("API_KEY_GOOGLE_REVIEW", "")

if not API_KEY:
    raise EnvironmentError(
        "⛔ API Key no encontrada.\n"
        "Crea un archivo .env con: GOOGLE_MAPS_API_KEY=AIzaSy_tu_clave"
    )

masked = API_KEY[:8] + "..." + API_KEY[-4:]
print(f"✓ API Key cargada: {masked}")

✓ API Key cargada: AIzaSyDl...OI70


/Users/fgoiri/Documents/Documentos_Fatima/Bootcamp/Desafio_Tripulaciones/iceberg/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuración

In [12]:
# ─── Rutas ────────────────────────────────────────────────────────────────────
INPUT_CSV    = "../data/processed/iceberg.csv"
OUTPUT_CSV   = "../data/processed/iceberg_reviews.csv"      # dataset final enriquecido
CACHE_FILE   = "../data/google_api_cache.json"        # caché: evita repetir peticiones
REVIEWS_FILE = "../data/processed/txoko_reviews_raw.json"       # reseñas en crudo para datos sintéticos

# ─── Umbrales haversine para google_match_conf ────────────────────────────────
THRESHOLD_HIGH   = 300   # metros — confianza ALTA   ✅
THRESHOLD_MEDIUM = 800   # metros — confianza MEDIA  ✅
#                          > 800m  — confianza BAJA  ❌ descartado

# ─── Pausa entre peticiones (cortesía a la API) ───────────────────────────────
SLEEP = 0.25   # segundos

print("✓ Configuración cargada")
print(f"  Umbrales: HIGH < {THRESHOLD_HIGH}m | MEDIUM < {THRESHOLD_MEDIUM}m")

✓ Configuración cargada
  Umbrales: HIGH < 300m | MEDIUM < 800m


## 3. Funciones auxiliares

In [13]:
def haversine_m(lat1, lon1, lat2, lon2):
    """Distancia en metros entre dos puntos GPS (fórmula haversine)."""
    R = 6_371_000
    p1, p2 = radians(lat1), radians(lat2)
    dp, dl = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dp/2)**2 + cos(p1)*cos(p2)*sin(dl/2)**2
    return 2*R*atan2(sqrt(a), sqrt(1-a))

def calc_conf(csv_lat, csv_lon, g_lat, g_lon):
    """Calcula google_match_conf comparando coords del CSV con las de Google."""
    if not all([csv_lat, csv_lon, g_lat, g_lon]):
        return "unknown"
    d = haversine_m(csv_lat, csv_lon, g_lat, g_lon)
    if   d < THRESHOLD_HIGH  : return "high"
    elif d < THRESHOLD_MEDIUM: return "medium"
    else                     : return "low"

def load_cache(p):
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else {}

def save_cache(p, c):
    json.dump(c, open(p,'w',encoding='utf-8'), ensure_ascii=False, indent=2)

# Test haversine
print(f"✓ Test haversine: mismo punto = {haversine_m(43.26,-2.93,43.26,-2.93):.1f}m")
print(f"✓ Test haversine: Bilbao→Donostia = {haversine_m(43.263,-2.935,43.318,-1.981)/1000:.0f}km")

✓ Test haversine: mismo punto = 0.0m
✓ Test haversine: Bilbao→Donostia = 77km


## 4. FASE 1 — Text Search (4.624 lugares)
**Obtiene:** `place_id`, `rating`, `num_reviews`, `geometry.location`  
**Coste:** ~$0.017/petición × 4.624 = **~$79**

In [14]:
def text_search(nombre, municipio, lat=None, lon=None):
    """
    Busca un lugar por nombre en Google Maps Places API (Text Search).
    
    Endpoint: GET /maps/api/place/textsearch/json
    Fields devueltos: place_id, rating, user_ratings_total, geometry.location
    
    Si tenemos coords del CSV, añadimos location bias para mejorar la precisión
    del match: Google prioriza resultados dentro del radio indicado.
    """
    params = {
        "query"   : f"{nombre} {municipio} Euskadi",
        "key"     : API_KEY,
        "language": "es",
    }
    if lat and lon:
        params["location"] = f"{lat},{lon}"
        params["radius"]   = 5000   # bias de 5km

    r = requests.get(
        "https://maps.googleapis.com/maps/api/place/textsearch/json",
        params=params, timeout=10
    )
    r.raise_for_status()
    data = r.json()

    if data.get("status") != "OK" or not data.get("results"):
        return None

    res = data["results"][0]
    return {
        "place_id"   : res.get("place_id"),
        "rating"     : res.get("rating"),
        "num_reviews": res.get("user_ratings_total"),
        "g_lat"      : res["geometry"]["location"]["lat"],
        "g_lon"      : res["geometry"]["location"]["lng"],
    }

print("✓ text_search() definida")

✓ text_search() definida


### Ejecutar Fase 1

In [20]:
df = pd.read_csv(INPUT_CSV)
df_work = df[df['lat'].notna()].copy()   # solo lugares con coordenadas
print(f"✓ Dataset: {len(df):,} total | {len(df_work):,} con coordenadas")
print(f"  Coste estimado Fase 1: ~${len(df_work)*0.017:.0f} USD")

cache = load_cache(CACHE_FILE)
ts_rows = []

for row in tqdm(df_work.itertuples(), total=len(df_work), desc="Fase 1 — Text Search"):
    ck = f"ts||{row.nombre}||{row.municipio}"

    if ck not in cache:
        try:
            cache[ck] = text_search(
                row.nombre, row.municipio,
                lat=row.lat if pd.notna(row.lat) else None,
                lon=row.lon if pd.notna(row.lon) else None,
            ) or {}
        except Exception as e:
            cache[ck] = {"error": str(e)}
        time.sleep(SLEEP)

    res  = cache[ck]
    conf = calc_conf(
        row.lat if pd.notna(row.lat) else None,
        row.lon if pd.notna(row.lon) else None,
        res.get("g_lat"), res.get("g_lon")
    ) if res.get("place_id") else "no_match"

    ts_rows.append({
        "id"                : row.id,
        "google_place_id"   : res.get("place_id"),
        "google_rating"     : res.get("rating"),
        "google_num_reviews": res.get("num_reviews"),
        "google_lat"        : res.get("g_lat"),
        "google_lon"        : res.get("g_lon"),
        "google_match_conf" : conf,
    })

    if len(ts_rows) % 200 == 0:
        save_cache(CACHE_FILE, cache)

save_cache(CACHE_FILE, cache)

df_ts = pd.DataFrame(ts_rows)
df_p1 = df_work.merge(df_ts, on="id", how="left")

# ── Resumen Fase 1 ─────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"FASE 1 COMPLETADA — {len(df_p1):,} lugares")
print(f"{'='*50}")
for conf in ["high","medium","low","no_match","unknown"]:
    n = (df_p1['google_match_conf']==conf).sum()
    print(f"  {conf:10s}: {n:4d} ({n/len(df_p1)*100:.1f}%)")
n_usable = df_p1['google_match_conf'].isin(['high','medium']).sum()
print(f"\n  Usables para Fase 2 (HIGH+MEDIUM): {n_usable} (~${n_usable*0.017:.0f} adicionales)")

✓ Dataset: 7,209 total | 6,060 con coordenadas
  Coste estimado Fase 1: ~$103 USD


Fase 1 — Text Search: 100%|██████████| 6060/6060 [59:31<00:00,  1.70it/s]


FASE 1 COMPLETADA — 6,060 lugares
  high      : 4640 (76.6%)
  medium    :  519 (8.6%)
  low       :  808 (13.3%)
  no_match  :   93 (1.5%)
  unknown   :    0 (0.0%)

  Usables para Fase 2 (HIGH+MEDIUM): 5159 (~$88 adicionales)


## 5. FASE 2 — Place Details (solo HIGH + MEDIUM)
**Obtiene:** hasta 5 reseñas por lugar  
**Coste:** ~$0.017/petición × ~2.800 lugares = **~$47**  
**Total acumulado: ~$126** ✅ dentro del crédito $200

In [21]:
def place_details(place_id):
    """
    Obtiene detalles de un lugar: rating verificado + hasta 5 reseñas.
    
    Endpoint: GET /maps/api/place/details/json
    
    Usamos el parámetro 'fields' para solicitar SOLO los campos necesarios.
    Esto es crucial para el coste:
      - Campos básicos (place_id, rating, reviews, geometry): ~$0.017 ← usamos estos
      - Campos avanzados (opening_hours, price_level, etc.): ~$0.040  ← evitamos
    
    Las reseñas se usan para generar datos sintéticos de entrenamiento,
    no para redistribuirlas ni publicarlas.
    """
    r = requests.get(
        "https://maps.googleapis.com/maps/api/place/details/json",
        params={
            "place_id": place_id,
            "key"     : API_KEY,
            "language": "es",
            # IMPORTANTE: solo campos del tier básico para no subir de precio
            "fields"  : "place_id,rating,user_ratings_total,reviews,geometry",
        },
        timeout=10
    )
    r.raise_for_status()
    data = r.json()

    if data.get("status") != "OK":
        return None

    res = data.get("result", {})
    reviews = [
        {
            "rating"     : rv.get("rating"),
            "text"       : rv.get("text", ""),
            "time"       : rv.get("time"),
            "author_name": rv.get("author_name",""),
            "language"   : rv.get("language",""),
        }
        for rv in res.get("reviews", [])
    ]
    return {
        "place_id"   : res.get("place_id"),
        "rating"     : res.get("rating"),
        "num_reviews": res.get("user_ratings_total"),
        "reviews"    : reviews,
    }

print("✓ place_details() definida")

✓ place_details() definida


### Ejecutar Fase 2

In [22]:
# Solo lugares con match HIGH o MEDIUM (los fiables)
df_fase2 = (df_p1[df_p1['google_match_conf'].isin(['high','medium','low'])]
            .dropna(subset=['google_place_id'])
            .copy())

print(f"Lugares para Fase 2: {len(df_fase2):,}")
print(f"Coste estimado Fase 2: ~${len(df_fase2)*0.017:.0f} USD")
print(f"Coste total acumulado: ~${(len(df_work) + len(df_fase2))*0.017:.0f} USD")

all_reviews = {}

for row in tqdm(df_fase2.itertuples(), total=len(df_fase2), desc="Fase 2 — Place Details"):
    ck = f"pd||{row.google_place_id}"

    if ck not in cache:
        try:
            cache[ck] = place_details(row.google_place_id) or {}
        except Exception as e:
            cache[ck] = {"error": str(e)}
        time.sleep(SLEEP)

    res = cache[ck]
    if res.get("reviews"):
        all_reviews[row.id] = res

    if len(all_reviews) % 200 == 0:
        save_cache(CACHE_FILE, cache)

save_cache(CACHE_FILE, cache)

# Guardar reseñas para el notebook de datos sintéticos
with open(REVIEWS_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_reviews, f, ensure_ascii=False, indent=2)

n_reviews = sum(len(v.get('reviews',[])) for v in all_reviews.values())
print(f"\n✓ Fase 2 completada")
print(f"  Lugares con reseñas: {len(all_reviews):,}")
print(f"  Total reseñas       : {n_reviews:,}")
print(f"  Promedio por lugar  : {n_reviews/max(1,len(all_reviews)):.1f}")
print(f"  Guardadas en        : {REVIEWS_FILE}")

Lugares para Fase 2: 5,967
Coste estimado Fase 2: ~$101 USD
Coste total acumulado: ~$204 USD


Fase 2 — Place Details: 100%|██████████| 5967/5967 [34:42<00:00,  2.87it/s]



✓ Fase 2 completada
  Lugares con reseñas: 5,365
  Total reseñas       : 26,106
  Promedio por lugar  : 4.9
  Guardadas en        : ../data/processed/txoko_reviews_raw.json


## 6. Guardar dataset final y resumen

In [23]:
# Añadir flag de si tiene reseñas
df_p1['has_reviews'] = df_p1['id'].isin(all_reviews.keys())

# Reordenar columnas
google_cols = ['google_place_id','google_rating','google_num_reviews',
               'google_lat','google_lon','google_match_conf','has_reviews']
orig_cols   = [c for c in df_p1.columns if c not in google_cols]
df_final    = df_p1[orig_cols + google_cols]

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f"✓ Dataset guardado: {OUTPUT_CSV}")
print(f"  {len(df_final):,} registros × {len(df_final.columns)} columnas")

print(f"\n{'='*55}")
print(f"RESUMEN EJECUTIVO — para el jurado")
print(f"{'='*55}")
print(f"  Fase 1 (Text Search)    : {len(df_work):,} peticiones  ~${len(df_work)*0.017:.0f}")
print(f"  Fase 2 (Place Details)  : {len(df_fase2):,} peticiones  ~${len(df_fase2)*0.017:.0f}")
print(f"  COSTE TOTAL ESTIMADO    : ~${(len(df_work)+len(df_fase2))*0.017:.0f} USD")
print(f"  Crédito gratuito Google : $200/mes")
print(f"  Margen restante         : ~${200-(len(df_work)+len(df_fase2))*0.017:.0f} USD")
print(f"\n  Con rating (usable)     : {df_final['google_rating'].notna().sum():,}")
print(f"  Con reseñas             : {df_final['has_reviews'].sum():,}")
print(f"  Match HIGH              : {(df_final['google_match_conf']=='high').sum():,}")
print(f"  Match MEDIUM            : {(df_final['google_match_conf']=='medium').sum():,}")
print(f"\n  Archivos generados:")
print(f"    {OUTPUT_CSV}")
print(f"    {REVIEWS_FILE}")
print(f"    {CACHE_FILE}")

✓ Dataset guardado: ../data/processed/iceberg_reviews.csv
  6,060 registros × 21 columnas

RESUMEN EJECUTIVO — para el jurado
  Fase 1 (Text Search)    : 6,060 peticiones  ~$103
  Fase 2 (Place Details)  : 5,967 peticiones  ~$101
  COSTE TOTAL ESTIMADO    : ~$204 USD
  Crédito gratuito Google : $200/mes
  Margen restante         : ~$-4 USD

  Con rating (usable)     : 5,416
  Con reseñas             : 5,365
  Match HIGH              : 4,640
  Match MEDIUM            : 519

  Archivos generados:
    ../data/processed/iceberg_reviews.csv
    ../data/processed/txoko_reviews_raw.json
    ../data/google_api_cache.json
